# CLaS-Bench: DiffMean Layer-by-Layer Language Steering Evaluation

This notebook executes the canonical CLaS-Bench scripts for **DiffMean** representation steering on `CohereLabs/aya-expanse-8b` across representative layers (`4, 12, 20, 28`) on English (`en`), German (`de`), and Spanish (`es`).

### Pipeline:
1. **Setup & Clone**: Prepares the repository.
2. **Dependencies**: Installs `vllm`, `transformers<5.0.0`, `bitsandbytes`, and FastText evaluation tools.
3. **Authentication**: Logs into Hugging Face for model access.
4. **Intervention**: Runs `2-diffmean-4.sh` (Aya Expanse DiffMean runner across layers 4, 12, 20, 28 with 30 questions).
5. **Evaluation**: Runs `X-evaluate-language-forcing-diffmean.sh` (FastText forcing success).
6. **Visualization**: Runs `B-vis-ablate-diffmean.sh` (generates multi-layer comparison plots and heatmaps).
7. **Download**: Archives results into a zip file for download.


In [ ]:
# 1. Environment & Repository Setup
import os

if not os.path.exists('/content/CLaS-Bench'):
    !git clone https://github.com/Trier-CLaS-SemEval-2027/CLaS-Bench.git /content/CLaS-Bench

%cd /content/CLaS-Bench/CLaS-Bench
!git checkout exploration
!git pull origin exploration
!pwd


In [ ]:
# 2. Install dependencies (vllm + 4-bit bitsandbytes + fasttext for evaluation)
!pip install -q "vllm==0.10.2" "transformers<5.0.0" bitsandbytes datasets fasttext-numpy2-wheel seaborn matplotlib


In [ ]:
# 3. Check GPU & Authenticate with Hugging Face
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from huggingface_hub import login
login()


In [ ]:
# 4. Run DiffMean Steering Intervention for Aya-Expanse (2-diffmean-4.sh)
# Sweeps layers 4, 12, 20, 28 with 30 questions per language pair in 4-bit quantization
%cd /content/CLaS-Bench/CLaS-Bench
!bash 2-diffmean-4.sh


In [ ]:
# 5. Evaluate Language Forcing Success Rate (X-evaluate-language-forcing-diffmean.sh)
%cd /content/CLaS-Bench/CLaS-Bench
!bash X-evaluate-language-forcing-diffmean.sh


In [ ]:
# 6. Visualize Layer-by-Layer DiffMean Results (B-vis-ablate-diffmean.sh)
%cd /content/CLaS-Bench/CLaS-Bench
!bash B-vis-ablate-diffmean.sh

from IPython.display import Image, display
import glob

# Display generated plots inline
plots = sorted(glob.glob("generation/diffmean/*.png"))
for p in plots:
    print(f"\nDisplaying: {p}")
    display(Image(filename=p))


In [ ]:
# 7. Package and Download Generated Results
!zip -r /content/generation_diffmean_layers.zip generation/

from google.colab import files
try:
    files.download('/content/generation_diffmean_layers.zip')
    print("Download initiated in browser!")
except Exception as e:
    print("Tip: If running in VS Code / IDE, right-click 'generation_diffmean_layers.zip' in the Colab files sidebar to download.")
